# 10주차 · 재구매 캠페인을 어느 고객에게 보낼까 — 군집분석

**⏱ 60분** · 영국 온라인 선물 도매상 고객 5,839명

## 🎯 이번 주의 질문

영국의 한 온라인 선물 도매상이 한동안 주문이 뜸한 고객에게 재구매 할인 쿠폰을 보내려 합니다. 쿠폰 예산은 한정되어 있어 모든 고객에게 보낼 수는 없습니다. 고객을 구매 행태가 비슷한 무리로 나누고, 그 가운데 어느 무리에 보낼지 정합니다. 정답 표시 없이 비슷한 대상끼리 묶는 분석을 **군집분석(clustering)** 이라고 합니다.

> **고객을 몇 개의 무리로 나누고, 재구매 캠페인은 어느 무리에 보낼까요?**

**이번 주에 할 일**
1. 주문 기록을 고객별 RFM 표로 요약한다 (1절)
2. 로그 변환과 표준화로 세 변수의 척도를 맞춘다 (2절)
3. k-means로 묶고, WSS로 군집 수를 정한다 (3~4절)
4. 계층 군집(Ward)으로 같은 고객을 묶어 결과를 비교한다 (5절)
5. 군집별 특징을 읽고 캠페인 대상을 정한다 (6~7절)

**데이터:** UCI Online Retail II. 영국 온라인 선물 도매상의 2009-12-01 ~ 2011-12-09 실제 주문 기록입니다(2주차와 같은 자료).

**반복할 패턴:** 대상별 요약표 만들기 → 척도 맞추기 → `KMeans` 학습 → WSS로 군집 수 정하기 → 군집별 요약으로 특징 읽기

배경이론: [10주차 이론](https://balab-pknu.github.io/bizanalytics/theory/week10.html)

| 셀 표시 | 할 일 |
|---|---|
| `# ✍️ 직접 입력` | 위에 보이는 코드를 직접 쳐서 실행합니다 |
| `# ▶ 그대로 실행하세요` | 준비·보조 코드입니다. 실행만 합니다 |
| `# 📝` | 과제입니다. 힌트를 보고 직접 씁니다 |

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 주문 기록을 읽고 정리합니다. (20초 안팎)
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", 500)       # 표를 줄임표(...) 없이 끝까지 보여 줌
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", None)
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

import os
import sys
for _p in (".", "..", "../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p); break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load
df = load("online_retail_ii").drop(columns=["Sheet"]).drop_duplicates()
df = df[df["Customer ID"].notna() & (df["Price"] > 0) & df["StockCode"].str.match(r"^\d{5}")]
cancel = df[df["Invoice"].str.startswith("C") & (df["Quantity"] < 0)].assign(Quantity=lambda d: -d["Quantity"])
df = df[~df["Invoice"].str.startswith("C") & (df["Quantity"] > 0)]
keys = ["Customer ID", "StockCode", "Quantity", "Price"]   # 취소 줄과 짝이 되는, 그 전에 산 주문 줄을 찾아 함께 뺌
pairs = pd.merge_asof(cancel.sort_values("InvoiceDate")[keys + ["InvoiceDate"]],
                      df.reset_index().sort_values("InvoiceDate")[keys + ["InvoiceDate", "index"]],
                      on="InvoiceDate", by=keys, direction="backward")
df = df.drop(index=pairs["index"].dropna().astype(int).unique())
df["amount"] = df["Quantity"] * df["Price"]
snapshot = pd.Timestamp("2011-12-10")
print("준비 완료!", df.shape)

✅ `준비 완료! (770593, 9)`가 나오면 됩니다. 한 행은 주문의 한 줄(주문 하나에 담긴 상품 하나)입니다. 고객 번호가 없는 줄, 수량·가격이 0 이하인 줄, 우편료·수수료처럼 상품이 아닌 줄은 뺐습니다. 취소 주문은 같은 고객·상품·수량·단가로 그 전에 산 주문 줄과 짝지어 함께 뺐습니다. `amount`는 그 줄의 금액(수량 × 단가, 파운드)이고, `snapshot`은 분석 기준일(마지막 주문 다음 날 2011-12-10)입니다.

---
## 1. 고객별 RFM 표

고객을 묶으려면 고객마다 한 행이 되도록 요약해야 합니다. 구매 행태를 요약하는 세 숫자를 **RFM**이라고 합니다.

| 지표 | 뜻 |
|---|---|
| R (recency) | 마지막 구매 후 기준일까지 지난 날수. 작을수록 최근 고객 |
| F (frequency) | 주문 수 |
| M (monetary) | 구매 금액 합계(파운드) |

고객마다 마지막 주문 시각을 구하고, 기준일에서 빼서 지난 날수를 셉니다. `.dt.days`는 기간을 날수로 바꿉니다.

```python
last = df.groupby("Customer ID")["InvoiceDate"].max()
recency = (snapshot - last).dt.days
recency.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 고객 12346은 마지막 구매가 529일 전, 12347은 2일 전입니다.

주문 수와 구매 금액 합계를 구해 세 열을 한 표로 모읍니다.

```python
frequency = df.groupby("Customer ID")["Invoice"].nunique()
monetary = df.groupby("Customer ID")["amount"].sum()
rfm = pd.DataFrame({"recency": recency, "frequency": frequency, "monetary": monetary})
rfm.shape
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `(5839, 3)`이면 맞습니다. 고객 5,839명, 열 3개입니다.

세 열의 분포를 봅니다.

```python
rfm.describe().round(1)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설**
- R은 0~738일, F는 1~369회, M은 3~579,129파운드입니다. 세 변수의 단위와 범위가 크게 다릅니다.
- M의 중앙값은 851파운드인데 평균은 2,830파운드, 최댓값은 58만 파운드입니다. 소수의 큰 고객이 평균을 끌어올리는 치우친 분포입니다. F도 중앙값 3회, 최댓값 369회로 같은 모양입니다.

---
## 2. 척도 맞추기

군집분석은 고객 사이의 **거리**로 가깝고 먼 것을 판단합니다. 두 고객의 거리는 R·F·M 차이를 각각 제곱해 더한 뒤 제곱근을 씌운 값(유클리드 거리)입니다. 첫 두 고객의 차이와 거리를 봅니다.

```python
diff = rfm.iloc[0] - rfm.iloc[1]
diff
```

In [ ]:
# ✍️ 위 코드를 입력하세요


```python
np.sqrt((diff ** 2).sum())
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 거리는 약 4,781입니다. R 차이 527일, F 차이 6회는 M 차이 4,752파운드에 묻혀 거리에 거의 반영되지 않습니다. 이대로 묶으면 구매 금액 하나로만 고객을 나누게 됩니다.

두 단계로 척도를 맞춥니다.
- **로그 변환** `np.log1p(x)` = log(1 + x). 큰 값의 차이를 줄여 치우침을 완화합니다. R이 0일인 고객이 있어 1을 더합니다.
- **표준화** (값 − 평균) ÷ 표준편차. 세 열을 모두 평균 0, 표준편차 1로 맞춥니다.

```python
X = np.log1p(rfm)
X = (X - X.mean()) / X.std()
X.describe().round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 세 열의 `mean`이 0.0, `std`가 1.0이면 맞습니다.

같은 두 고객의 거리를 다시 잽니다.

```python
np.sqrt(((X.iloc[0] - X.iloc[1]) ** 2).sum())
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 거리가 4.28입니다. 이제 R(3.25), F(1.36), M(2.44)의 표준화된 차이가 모두 거리에 반영됩니다.

---
## 3. k-means로 묶기

**k-means**는 군집 수 k를 정해 주면, 각 고객을 가장 가까운 군집 중심(**centroid**)에 배정하고 중심을 다시 계산하는 일을 배정이 바뀌지 않을 때까지 반복합니다. 결과는 시작 중심에 따라 달라질 수 있어, `n_init=10`으로 시작점을 10번 바꿔 돌린 뒤 가장 좋은 결과를 남깁니다. `random_state=0`은 매번 같은 결과가 나오게 합니다. 먼저 k = 4로 묶어 봅니다.

```python
model = KMeans(n_clusters=4, n_init=10, random_state=0)
model.fit(X)
model.labels_[:10]
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 `labels_`는 고객마다 배정된 군집 번호(0~3)입니다. 번호는 이름표일 뿐 크고 작음의 뜻이 없습니다.

군집마다 고객 수를 셉니다.

```python
pd.Series(model.labels_).value_counts()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 네 군집의 크기가 1,161 · 1,232 · 1,468 · 1,978명이면 맞습니다(번호 순서는 달라도 됩니다).

각 고객과 자기 군집 중심 사이 거리의 제곱을 모두 더한 값을 **WSS**(within-cluster sum of squares)라고 합니다. 작을수록 군집 안의 고객이 서로 가깝습니다. `inertia_`에 들어 있습니다.

```python
model.inertia_
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 WSS는 약 4,926입니다.

---
## 4. 군집 수 정하기

k를 1부터 8까지 바꿔 가며 WSS를 구합니다.

```python
wss = []
for k in range(1, 9):
    model_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    wss.append(model_k.inertia_)
pd.Series(wss, index=range(1, 9)).round(1)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


In [ ]:
# ▶ 그대로 실행하세요 — k에 따른 WSS 그림
plt.figure(figsize=(6, 3.5))
plt.plot(range(1, 9), wss, marker="o")
plt.xlabel("Number of clusters k")
plt.ylabel("WSS")
plt.title("WSS by number of clusters")
plt.show()

💬 k를 늘리면 WSS는 늘 줄어듭니다. 군집을 많이 만들수록 중심이 고객에게 가까워지기 때문입니다. 그래서 WSS가 가장 작은 k가 아니라, **군집을 하나 늘려도 WSS가 별로 줄지 않기 시작하는 k**를 고릅니다. 줄어드는 비율을 봅니다. `pct_change()`는 바로 앞 값 대비 변화율입니다.

```python
(pd.Series(wss, index=range(1, 9)).pct_change() * 100).round(1)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ k = 2·3·4에서 −51.2 · −25.9 · −22.2(%), k = 5부터 −16.7 · −13.3 · −10.4 · −9.2(%)이면 맞습니다.

💬 **결과 해설**
- k = 4까지는 군집을 하나 늘릴 때마다 WSS가 20% 이상 줄고, k = 5부터는 17% 이하로 줄어듭니다. 이 기준으로 **k = 4**를 씁니다.
- 6절에서 보듯 k = 4에서 결정에 필요한 고객군(한동안 사지 않았지만 매출이 있던 고객)이 따로 묶입니다. 군집 수는 숫자 기준과 결정에 쓸 수 있는지를 함께 보고 정합니다.

---
## 5. 계층 군집으로 비교

**계층 군집**은 고객 한 명 한 명을 각각 하나의 군집으로 두고, 가장 가까운 두 군집을 합치는 일을 하나가 남을 때까지 반복합니다. **Ward** 방식은 합쳤을 때 WSS가 가장 적게 늘어나는 두 군집을 합칩니다. 군집 수를 미리 정하지 않고, 합친 순서를 나무 모양 그림(**dendrogram**)으로 남깁니다.

```python
Z = linkage(X, method="ward")
Z.shape
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 `Z`의 한 행이 합치기 한 번입니다. 고객 5,839명을 하나로 합치기까지 5,838번 합칩니다. 마지막 12번의 합치기만 그림으로 봅니다.

In [ ]:
# ▶ 그대로 실행하세요 — 마지막 12개 군집의 dendrogram
plt.figure(figsize=(8, 3.5))
dendrogram(Z, truncate_mode="lastp", p=12)
plt.xlabel("Cluster (number of customers)")
plt.ylabel("Ward distance at merge")
plt.show()

💬 가로축 아래 괄호 안 숫자는 그 가지에 든 고객 수, 세로축은 두 군집을 합칠 때의 거리입니다. 위로 갈수록 서로 먼 군집을 합친 것입니다. 가로로 선을 그어 자르면 선 아래에 남은 가지 수가 군집 수입니다.

dendrogram을 군집 4개로 잘라 k-means 결과와 표로 비교합니다. `fcluster`의 `maxclust`는 군집이 4개가 되도록 자릅니다.

```python
labels_ward = fcluster(Z, t=4, criterion="maxclust")
pd.crosstab(model.labels_, labels_ward)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설**
- 행은 k-means 군집, 열은 Ward 군집입니다. 행마다 가장 큰 칸을 더하면 998 + 1,013 + 1,707 + 1,108 = 4,826명, 83%가 두 방법에서 같은 무리에 들어갑니다.
- 크게 사는 고객과 오래전에 한 번 산 고객은 두 방법이 거의 같게 묶습니다.
- k-means의 1,468명 군집(한동안 사지 않은 중간 고객)은 1,013명(69%)만 Ward의 한 군집에 함께 들어가고 나머지는 다른 군집으로 갈립니다. 군집의 경계는 방법에 따라 달라질 수 있습니다.

---
## 6. 군집별 특징 읽기

k-means 결과를 고객 표에 붙이고, 군집마다 중앙값을 봅니다. 치우친 분포라 평균보다 중앙값이 대표값으로 낫습니다.

```python
rfm["cluster"] = model.labels_
rfm.groupby("cluster").median().round(1)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


군집마다 고객 수와 매출 비중을 봅니다.

```python
summary = rfm.groupby("cluster").agg(customers=("recency", "size"), sales=("monetary", "sum"))
summary["sales_share"] = (summary["sales"] / summary["sales"].sum()).round(3)
summary
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 매출 비중이 가장 큰 군집이 약 0.731(고객 1,161명)이면 맞습니다.

💬 **결과 해설** — 네 군집에 특징으로 이름을 붙입니다.

| 이름 | 고객 수 | 마지막 구매 | 주문 수 | 구매 금액 | 매출 비중 |
|---|---|---|---|---|---|
| 우수 고객 | 1,161명 | 16일 전 | 13회 | 4,922 | 73% |
| 최근 소액 고객 | 1,232명 | 23일 전 | 3회 | 717 | 6% |
| 이탈 위험 고객 | 1,468명 | 178일 전 | 5회 | 1,455 | 17% |
| 이탈 고객 | 1,978명 | 400일 전 | 1회 | 270 | 4% |

- 우수 고객은 최근에도 자주 사고, 고객의 20%가 매출의 73%를 냅니다.
- 이탈 위험 고객은 주문 5회, 1,455파운드로 구매 이력은 있지만 마지막 구매가 여섯 달 가까이 전입니다.
- 이탈 고객은 1년 넘게 전에 한 번 산 고객입니다.

---
## 7. 결정

> **재구매 쿠폰은 이탈 위험 고객 1,468명에게 보낸다 (마지막 구매 중앙값 178일 전, 주문 5회, 매출 비중 17%).**

- 우수 고객과 최근 소액 고객은 최근에 사고 있어 쿠폰 없이도 다시 살 가능성이 큽니다.
- 이탈 고객은 1년 넘게 전에 한 번 산 고객이라 쿠폰 한 장으로 되돌리기 어렵습니다.
- 이탈 위험 고객은 구매 이력이 있고 아직 1년이 지나지 않았습니다. 다만 Ward 결과와 경계가 다르므로, 쿠폰의 효과는 이 군집 안에서 무작위로 일부에게만 보내 비교해야 압니다.

---
## 8. 실습 과제

### 📝 과제 1 · 군집 수 3개

3절과 6절을 k = 3으로 다시 해 보세요. 이탈 위험 고객이 따로 묶이나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 3절의 KMeans에서 n_clusters만 3으로 바꿔 학습하고, 6절의 groupby 중앙값과 고객 수를 봅니다.


✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

### 📝 과제 2 · 로그 변환 없이 표준화만

2절에서 로그 변환을 빼고 표준화만 한 표로 k = 4 k-means를 돌려 군집 크기를 보세요. 결과가 결정에 쓸 만한가요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 2절의 표준화 식을 rfm[["recency", "frequency", "monetary"]]에 바로 쓰고, 3절처럼 학습한 뒤 value_counts를 봅니다.


✏️ **나의 답:**

### 📝 과제 3 · average linkage

5절의 계층 군집을 Ward 대신 average linkage(두 군집 고객 사이 거리의 평균)로 바꿔 4개로 잘라 크기를 보세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: linkage의 method를 "average"로 바꾸고, fcluster로 4개로 자른 뒤 pd.Series(...).value_counts()로 셉니다.


✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| RFM | 고객별 최근성·빈도·금액. 고객 5,839명 × 3열 |
| 척도 맞추기 | 로그 변환 → 표준화. 첫 두 고객의 거리 4,781 → 4.28 |
| k-means | 가까운 중심에 배정 → 중심 갱신을 반복. `KMeans(n_clusters, n_init, random_state)` |
| WSS로 k 정하기 | k를 늘릴 때 WSS 감소율이 20% 이상인 마지막 k = 4 |
| 계층 군집(Ward) | 가장 가까운 두 군집을 합치기를 반복, dendrogram을 잘라 군집 수를 정함. k-means와 83% 일치 |
| 군집 해석 | 군집 번호 대신 중앙값·크기·매출 비중으로 이름을 붙이고 결정에 연결 |